In [ ]:
from db import get_con
import plotly.express as px

In [ ]:
con = get_con()

# Data exploration

## Part 1: Studying the events that occurred, to understand how they affect the data

In [5]:
con.sql(
    """
    SELECT *
    FROM business_events
    ORDER BY date(event_date)
    """).df()

,event_date,event_type,affected_product,affected_customer_id,description
0,2025-07-01,pricing_change,compute_standard,None,Compute list price reduced for new and renewin...
1,2025-10-01,product_migration,gpu_a100,None,Legacy GPU SKU migrated to gpu_accelerated in ...
2,2026-03-15,capacity_constraint,gpu_accelerated,None,Temporary GPU supply constraint delayed some e...
3,2026-04-04,billing_system_change,NaN,None,Invoice export switched to a new source table;...
4,2026-05-20,sales_policy,NaN,None,Sales leadership asked teams to split saving-p...


Main events
1. pricing_change (2025-07) — a price change. Important: the price change affects only **new** or **renewing** contracts.
2. gpu_a100 (2025-10) — just a rename of the old name; info used to collapse different products into a single product
3. capacity_constraint (2026-03) — a memory constraint. March turns out to be an unrepresentative month.
4. billing_system_change (2026-04) — a new billing system. Duplicates are possible, needs careful study
5. sales_policy (2026-05) — new data-recording logic. Need to look at crm_opportunity separately — how the new data is recorded

In [59]:
con.sql(
    """
    SELECT *
    FROM price_list
    """).df()

,product,effective_from,unit,list_price_usd,notes
0,compute_standard,2024-01-01,compute_hour,0.046,General purpose VM compute.
1,gpu_a100,2024-01-01,gpu_hour,2.650,Legacy GPU SKU used before product migration.
2,gpu_accelerated,2025-10-01,gpu_hour,2.550,Renamed GPU SKU after platform migration.
3,object_storage,2024-01-01,tb_month_avg,22.000,Average stored TB per month.
4,network_egress,2024-01-01,egress_tb,74.000,Public internet egress.
5,compute_standard,2025-07-01,compute_hour,0.043,Mid-2025 price decrease for compute.
6,object_storage,2026-01-01,tb_month_avg,20.500,Storage price refresh.


There were several price changes. But there was one price change described in the business events — compute_standard on 2025-07-01, and it does not apply to everyone.

## Part 2: Studying monthly_usage

In [10]:
con.sql(
    """
    SELECT *
    FROM monthly_usage
    ORDER BY date(month) desc,
             customer_id,
             product
    """).df()

,month,customer_id,product,compute_hours,gpu_hours,storage_tb_avg,egress_tb,active_projects,active_users
0,2026-05-01,CUST-0001,compute_standard,72282.16,0.00,0.00,0.00,34,231
1,2026-05-01,CUST-0001,gpu_accelerated,0.00,2291.94,0.00,0.00,38,246
2,2026-05-01,CUST-0001,network_egress,0.00,0.00,0.00,122.13,36,225
3,2026-05-01,CUST-0001,object_storage,0.00,0.00,422.24,0.00,38,231
4,2026-05-01,CUST-0002,compute_standard,3136.15,0.00,0.00,0.00,2,9
...,...,...,...,...,...,...,...,...,...
8506,2024-01-01,CUST-0087,object_storage,0.00,0.00,50.22,0.00,4,52
8507,2024-01-01,CUST-0090,compute_standard,3915.53,0.00,0.00,0.00,6,8
8508,2024-01-01,CUST-0090,gpu_a100,0.00,55.90,0.00,0.00,2,19
8509,2024-01-01,CUST-0090,network_egress,0.00,0.00,0.00,44.69,2,16


This table contains the main usage information.
The information is aggregated by
- Month (month)
- Customer (customer_id)
- Product (product)

In addition
- Each product has its own metric; for the other products it will be 0.00. We need to reduce it to a single metric for convenience
- There are also additional metrics: active_projects and active_users, but they don't really affect billing

In [27]:
con.sql(
    """
    SELECT count(distinct customer_id) as cnt_customers,
           count(distinct if(date(month) = date('2026-05-01'), customer_id, null)) as cnt_customers_last_month,
           count(*) as cnt_rows,
           count(concat(cast(month as string),
                        customer_id,
                        product)) as cnt_all_metrics
    FROM monthly_usage
    """).df()

,cnt_customers,cnt_customers_last_month,cnt_rows,cnt_all_metrics
0,90,83,8511,8511


1. 90 customers over the whole history
2. 83 customers in the last month
3. No empty rows

In [19]:
df = con.sql(
    """
    SELECT date(month) as pmonth,
           case product
           when 'compute_standard' then 'compute_standard'
           when 'gpu_a100' then 'gpu'
           when 'gpu_accelerated' then 'gpu'
           when 'network_egress' then 'network_egress'
           when 'object_storage' then 'object_storage'
           end as product,

           sum(case product
               when 'compute_standard' then compute_hours
               when 'gpu_a100' then gpu_hours
               when 'gpu_accelerated' then gpu_hours
               when 'network_egress' then egress_tb
               when 'object_storage' then storage_tb_avg
               end) as quantity
    FROM monthly_usage
    GROUP BY 1,2
    ORDER BY 1,2
    """).df()

In [20]:
fig = px.bar(
    df,
    x="pmonth",
    y="quantity",
    color="product",
    barmode="stack",
    title="Product usage per month",
    labels={"product": "Products", "pmonth": "Month", "quantity": "Product Metrics"}
)
# fig.write_image("block1_customers_by_segment.png")
fig.show()

In [21]:
df = con.sql(
    """
    SELECT date(month) as pmonth,
           sum(if(product = 'compute_standard', compute_hours, 0)) as compute_standard,
           sum(if(product in ('gpu_a100', 'gpu_accelerated'), gpu_hours, 0)) as gpu_hours,
           sum(if(product = 'network_egress', egress_tb, 0)) as egress_tb,
           sum(if(product = 'object_storage', storage_tb_avg, 0)) as storage_tb_avg
    FROM monthly_usage
    GROUP BY 1
    ORDER BY 1
    """).df()

In [22]:
fig = px.bar(
    df,
    x="pmonth",
    y="compute_standard",
    title="Compute Standard Dynamic",
    labels={"pmonth": "Month"}
)
# fig.write_image("block1_customers_by_segment.png")
fig.show()

In [23]:
fig = px.bar(
    df,
    x="pmonth",
    y="gpu_hours",
    title="GPU Dynamic",
    labels={"pmonth": "Month"}
)
# fig.write_image("block1_customers_by_segment.png")
fig.show()

In [25]:
fig = px.line(
    df,
    x="pmonth",
    y=["egress_tb", "storage_tb_avg"],
    title="Egress & Storage Dynamic",
    labels={"pmonth": "Month"}
)
# fig.write_image("block1_customers_by_segment.png")
fig.show()

What we see in the overall dynamics
1. All products grow year over year
2. There are hints of seasonality, but it is fairly unstable
3. GPU grew strongly specifically in November 2025 — a sharp jump there

In [28]:
df = con.sql(
    """
    with cte_prep_data as
             (SELECT date(month) as pmonth,
                     customer_id,
                     case product
                     when 'compute_standard' then 'compute_standard'
                     when 'gpu_a100' then 'gpu'
                     when 'gpu_accelerated' then 'gpu'
                     when 'network_egress' then 'network_egress'
                     when 'object_storage' then 'object_storage'
                     end as product,

                     sum(case product
                         when 'compute_standard' then compute_hours
                         when 'gpu_a100' then gpu_hours
                         when 'gpu_accelerated' then gpu_hours
                         when 'network_egress' then egress_tb
                         when 'object_storage' then storage_tb_avg
                         end) as quantity
              FROM monthly_usage
              GROUP BY 1,2,3)

        , cte_products_with_customers as
        (select pmonth,
                customer_id,
                count(distinct product) as cnt_products
         from cte_prep_data
         where quantity > 0
         group by 1,2)

    select pmonth,
           cnt_products,
           count(distinct customer_id) as cnt_customers
    from cte_products_with_customers
    group by 1,2
    order by 1,2
    """).df()

In [29]:
fig = px.line(
    df,
    x="pmonth",
    y="cnt_customers",
    color="cnt_products",
    title="Count of Customers with products",
    labels={"cnt_products": "Count of Products", "pmonth": "Month", "cnt_customers": "Count of Customers"}
)
# fig.write_image("block1_customers_by_segment.png")
fig.show()

Interesting points here
1. Most customers have all products
2. But there is a small number of customers who have only 3 of the 4 products. A small number of customers have no GPU.
3. Also on the overall dynamics: until July 1, 2025 — steady growth. After that the number of active customers dropped slightly, but starts to recover.

In [33]:
df = con.sql(
    """
    SELECT date(month) as pmonth,
           customer_id,
           sum(if(product = 'compute_standard', compute_hours, 0)) as compute_standard,
           sum(if(product in ('gpu_a100', 'gpu_accelerated'), gpu_hours, 0)) as gpu_hours,
           sum(if(product = 'network_egress', egress_tb, 0)) as egress_tb,
           sum(if(product = 'object_storage', storage_tb_avg, 0)) as storage_tb_avg
    FROM monthly_usage
    GROUP BY 1, 2
    ORDER BY 1, 2
    """).df()

In [34]:
fig = px.line(
    df,
    x="pmonth",
    y="compute_standard",
    color="customer_id",
    title="Compute Standard per Customer"
)
# fig.write_image("block1_customers_by_segment.png")
fig.show()

Here there are clear customers that are larger than all the others
1. Cust-0006 (Top-1 partner)

Examples of large partners, but smaller than top-1
1. Cust-0024
2. Cust-0069
3. Cust-0001


In [36]:
fig = px.line(
    df,
    x="pmonth",
    y="gpu_hours",
    color="customer_id",
    title="GPU per Customer"
)
# fig.write_image("block1_customers_by_segment.png")
fig.show()

1. There is a customer that has been growing since November 2025. It is what creates the incredibly large growth
2. There is a one-off outlier for one customer, cust-0033

## Part 3: How the other tables relate to the usage table

First, let's briefly describe the tables we want to match against the usage table

In [39]:
con.sql(
    """
    SELECT *
    FROM contracts
    """).df()

,contract_id,customer_id,contract_start,contract_end,committed_spend_usd,billing_frequency,discount_pct,payment_terms_days,auto_renewal,contract_status
0,CON-1000,CUST-0001,2024-09-01,2027-08-31,1794526,annual_prepay,15,45,True,active
1,CON-1001,CUST-0002,2024-10-01,2025-09-30,23826,quarterly,5,60,False,active
2,CON-1002,CUST-0004,2025-12-01,2026-11-30,37639,annual_prepay,0,30,False,active
3,CON-1003,CUST-0005,2024-04-01,2025-03-31,578345,quarterly,18,60,False,active
4,CON-1004,CUST-0006,2024-01-01,2026-12-31,1133894,monthly,15,14,False,active
5,CON-1005,CUST-0007,2025-05-01,2026-04-30,3432982,quarterly,0,45,True,active
6,CON-1006,CUST-0008,2024-07-01,2025-06-30,824818,annual_prepay,12,30,False,active
7,CON-1007,CUST-0009,2024-07-01,2025-06-30,393303,annual_prepay,12,45,False,active
8,CON-1008,CUST-0010,2024-03-01,2025-02-28,55694,quarterly,10,14,True,active
9,CON-1009,CUST-0013,2025-07-01,2027-06-30,101973,annual_prepay,10,14,False,active


1. There are different contracts in terms of their duration
2. There are contracts with and without auto-renewal
3. There is different logic built into the payment
4. There are contracts in different statuses: active and not
5. The contract start date and contract end date are specified
6. We can also use customer_id for the join

It is also immediately clear that there are fewer contracts than customers who use the service.

In [42]:
con.sql(
    """
    SELECT customer_id,
           count(distinct contract_id) as cnt_contracts,
           count(*) as cnt_rows
    FROM contracts
    group by customer_id
    order by 3 desc
    """).df()

,customer_id,cnt_contracts,cnt_rows
0,CUST-0058,2,2
1,CUST-0019,1,1
2,CUST-0073,1,1
3,CUST-0024,1,1
4,CUST-0033,1,1
5,CUST-0061,1,1
6,CUST-0088,1,1
7,CUST-0004,1,1
8,CUST-0048,1,1
9,CUST-0059,1,1


In [60]:
con.sql(
    """
    SELECT *
    FROM contracts
    where customer_id = 'CUST-0058'
    """).df()

,contract_id,customer_id,contract_start,contract_end,committed_spend_usd,billing_frequency,discount_pct,payment_terms_days,auto_renewal,contract_status
0,CON-1027,CUST-0058,2024-11-01,2025-10-31,1292997,quarterly,15,30,False,active
1,CON-1888,CUST-0058,2026-06-01,2027-05-31,1800000,annual_prepay,20,45,True,signed_not_started


There is only 1 customer with 2 contracts.
CUST-0058.
They had 1 contract in effect, without auto-renewal, for 1 quarter. Without auto-renewal.

Interestingly, the old contract is listed as active. This makes it look like the contract status cannot be fully trusted.


In [44]:
con.sql(
    """
    SELECT count(distinct mu.customer_id) as cnt_mu_customers,
           count(distinct cnt.customer_id) as cnt_cnt_customers,
           count(distinct if(cnt.customer_id is null, mu.customer_id, null)) as cnt_mu_customers_without_cnt,
           count(distinct if(mu.customer_id is null, cnt.customer_id, null)) as cnt_cnt_customers_without_mu
    FROM monthly_usage mu
    full join contracts cnt
           on mu.customer_id = cnt.customer_id
    """).df()

,cnt_mu_customers,cnt_cnt_customers,cnt_mu_customers_without_cnt,cnt_cnt_customers_without_mu
0,90,45,45,0


1. All customers with contracts have some usage records
2. But customers who have usage but no contract — 45 customers

In [47]:
con.sql(
    """
    with cte_usage_info as
        (select customer_id,
                max(month) as last_month
         from monthly_usage
         group by 1)

        , cte_contracts_info as
        (select distinct
                customer_id,
                date_trunc('month', contract_end) as contract_end,
                auto_renewal,
                contract_status
         from contracts)

    select contract_end,
           contract_status,
           auto_renewal,
           last_month,
           count(distinct cci.customer_id) as cnt_customers
    from cte_contracts_info cci
    left join cte_usage_info cui
        on cci.customer_id = cui.customer_id
    group by 1,2,3,4
    order by 1,2,3,4
    """).df()

,contract_end,contract_status,auto_renewal,last_month,cnt_customers
0,2024-12-01,active,True,2026-05-01,1
1,2025-01-01,active,False,2026-05-01,3
2,2025-02-01,active,True,2026-05-01,1
3,2025-03-01,active,False,2026-05-01,2
4,2025-03-01,active,True,2026-05-01,1
5,2025-06-01,active,False,2026-05-01,3
6,2025-09-01,active,False,2026-05-01,1
7,2025-10-01,active,False,2026-05-01,2
8,2025-11-01,active,True,2026-05-01,3
9,2025-12-01,active,False,2026-05-01,2


We make the following assumptions for the future recalculation
1. If there is no contract, then there is some general post-payment system and common average rules for when to pay
2. If the contract has ended, the logic is the same
3. If the contract's date has passed and there is no auto_renewal — we consider the contract ended

In [48]:
con.sql(
    """
    select *
    from billing_customers
    """).df()

,customer_id,customer_name,segment,industry,region,account_owner,signup_date,billing_country,payment_terms_days,customer_status
0,CUST-0001,Acme Analytics,Strategic,E-commerce,Middle East,M. Jensen,2022-11-01,PL,45,active
1,CUST-0002,HelioBank,SMB,AI/ML,APAC,M. Jensen,2025-04-01,DE,60,active
2,CUST-0003,Vertex Labs,Commercial,Healthcare,Europe,A. Novak,2022-07-01,FR,45,active
3,CUST-0004,Northstar Gaming,SMB,E-commerce,Europe,T. Meyer,2022-05-01,NL,30,paused
4,CUST-0005,BlueRiver AI,Enterprise,AI/ML,Europe,A. Novak,2023-10-01,GB,60,active
...,...,...,...,...,...,...,...,...,...,...
85,CUST-0086,CloudNest Technologies,SMB,SaaS,APAC,S. Patel,2024-09-01,AE,60,active
86,CUST-0087,Finwise Technologies,Commercial,AI/ML,North America,T. Meyer,2023-03-01,FR,14,active
87,CUST-0088,Orbit Media Technologies,Enterprise,E-commerce,APAC,L. Garcia,2024-04-01,AE,60,active
88,CUST-0089,Quantum Cart Technologies,Commercial,Media,North America,N. Kowalski,2024-06-01,GB,45,active


In [49]:
con.sql(
    """
    select customer_id,
           count(*) as cnt_rows
    from billing_customers
    group by customer_id
    order by 2 desc
    """).df()

,customer_id,cnt_rows
0,CUST-0014,1
1,CUST-0064,1
2,CUST-0048,1
3,CUST-0053,1
4,CUST-0059,1
...,...,...
85,CUST-0037,1
86,CUST-0040,1
87,CUST-0045,1
88,CUST-0046,1


1. The billing-customers table also contains information about the customer status
2. There is also information about how many days there are between issuing the invoice and the payment date
3. 1 customer — 1 record

In [51]:
con.sql(
    """
    with cte_usage_info as
        (select customer_id,
                max(month) as last_month
         from monthly_usage
         group by 1)

        , cte_billing_info as
        (select customer_id,
                customer_status
         from billing_customers)

    select customer_status,
           last_month,
           count(distinct cci.customer_id) as cnt_customers
    from cte_usage_info cci
    left join cte_billing_info cbi
        on cci.customer_id = cbi.customer_id
    group by 1,2
    order by 1,2
    """).df()

,customer_status,last_month,cnt_customers
0,active,2026-05-01,79
1,churned,2025-10-01,1
2,churned,2026-01-01,1
3,churned,2026-02-01,1
4,churned,2026-05-01,4
5,paused,2026-01-01,4


1. The customer status looks like a much more reliable indicator.
2. And at the same time — we have 4 churned customers whose last usage date is 2026-05-01. Let's assume these customers have left.

In [52]:
con.sql(
    """
    select *
    from invoices
    """).df()

,invoice_id,customer_id,invoice_date,service_month,contract_id,invoice_type,amount_usd,due_date,paid_date,payment_status
0,INV-70000,CUST-0001,2024-02-04,2024-01-01,NaN,usage,17405.99,2024-03-20,2024-03-20,paid
1,INV-70001,CUST-0001,2024-03-04,2024-02-01,NaN,usage,18012.21,2024-04-18,2024-04-18,paid
2,INV-70002,CUST-0001,2024-04-02,2024-03-01,NaN,usage,19299.41,2024-05-17,2024-06-04,paid
3,INV-70003,CUST-0001,2024-05-04,2024-04-01,NaN,usage,17041.90,2024-06-18,2024-07-06,paid
4,INV-70004,CUST-0001,2024-06-05,2024-05-01,NaN,usage,18738.35,2024-07-20,2024-08-07,paid
...,...,...,...,...,...,...,...,...,...,...
2157,INV-72155,CUST-0090,2026-06-05,2026-05-01,NaN,usage,19099.20,2026-08-04,2026-08-11,paid
2158,INV-72156,CUST-0018,2026-04-05,2026-03-01,NaN,service_credit,-215000.00,2026-04-05,2026-04-05,paid
2159,INV-72157,CUST-0033,2026-05-06,2026-04-01,NaN,billing_correction,-84000.00,2026-05-06,2026-05-06,paid
2160,INV-72158,CUST-0007,2026-01-05,2025-12-01,NaN,sla_credit,-36000.00,2026-01-05,2026-01-05,paid


## Part 4: CRM tables and the actual relationship

In [56]:
con.sql(
    """
    select *
    from crm_accounts
    """).df()

,crm_account_id,account_name,parent_account_name,segment,industry,region,account_owner,matched_customer_id,match_confidence
0,ACC-0001,Acme Analytics,NaN,Strategic,E-commerce,Middle East,M. Jensen,CUST-0001,low
1,ACC-0002,HelioBank,NaN,SMB,AI/ML,APAC,M. Jensen,CUST-0002,high
2,ACC-0003,Vertex Labs,NaN,Commercial,Healthcare,Europe,A. Novak,CUST-0003,medium
3,ACC-0004,Northstar Gaming,NaN,SMB,E-commerce,Europe,T. Meyer,CUST-0004,high
4,ACC-0005,BlueRiver AI,NaN,Enterprise,AI/ML,Europe,A. Novak,CUST-0005,medium
...,...,...,...,...,...,...,...,...,...
74,ACC-0075,NovaLearn Cloud,NaN,Commercial,Media,APAC,L. Garcia,CUST-0075,high
75,ACC-0091,Lambda Robotics,NaN,Enterprise,E-commerce,APAC,L. Garcia,NaN,new_logo
76,ACC-0092,Magellan BioAI,NaN,Strategic,Healthcare,APAC,S. Patel,NaN,new_logo
77,ACC-0093,Pinnacle Training,NaN,Strategic,AI/ML,Europe,A. Novak,NaN,new_logo


In [57]:
con.sql(
    """
    select *
    from crm_opportunities
    """).df()

,opportunity_id,crm_account_id,opportunity_name,opportunity_type,stage,forecast_category,probability,expected_close_date,expected_start_month,contract_term_months,total_contract_value_usd,expected_committed_spend_usd,expected_monthly_consumption_usd,payment_structure,owner_forecast_notes
0,OPP-1001,ACC-0002,HelioBank expansion gpu_reserved_capacity,New Logo,Solution Fit,Upside,0.55,2026-08-24,2026-09-01,24,229146,126273,11975,commit_drawdown,Legal review in progress.
1,OPP-1002,ACC-0008,Orbit Media expansion compute_savings_plan,Renewal,Solution Fit,Upside,0.40,2026-08-14,2026-09-01,24,646769,610923,29852,quarterly_prepay,Legal review in progress.
2,OPP-1003,ACC-0009,Quantum Cart expansion compute_savings_plan,New Logo,Proposal,Best Case,0.25,2026-08-22,2026-10-01,24,76206,62452,2799,monthly_arrears,Sales expects fast ramp.
3,OPP-1004,ACC-0012,BrightApps expansion cloud_commit_plan,Expansion,Negotiation,Commit,0.70,2026-08-16,2026-09-01,24,543070,422515,15928,commit_drawdown,Sales expects fast ramp.
4,OPP-1005,ACC-0014,Atlas Retail expansion gpu_reserved_capacity,Expansion,Solution Fit,Pipeline,0.25,2026-07-15,2026-07-01,36,411830,246629,24492,annual_prepay,Sales expects fast ramp.
5,OPP-1006,ACC-0015,NovaLearn expansion network_commit,Expansion,Discovery,Pipeline,0.25,2026-10-25,2026-11-01,36,814034,762592,39798,quarterly_prepay,Legal review in progress.
6,OPP-1007,ACC-0016,PeakMetrics expansion cloud_commit_plan,New Logo,Proposal,Commit,0.55,2026-07-09,2026-09-01,36,630226,447413,26980,quarterly_prepay,Procurement timing uncertain.
7,OPP-1008,ACC-0019,OmniPay expansion gpu_reserved_capacity,New Logo,Proposal,Best Case,0.70,2026-09-22,2026-10-01,12,723746,424568,41603,monthly_arrears,Ramp depends on capacity allocation.
8,OPP-1009,ACC-0022,HelioBank Ltd. expansion gpu_reserved_capacity,Expansion,Commit,Best Case,0.40,2026-08-15,2026-08-01,24,428490,328326,18694,monthly_arrears,Sales expects fast ramp.
9,OPP-1010,ACC-0024,Northstar Gaming Ltd. expansion object_storage...,New Logo,Proposal,Pipeline,0.25,2026-09-14,2026-09-01,36,1097100,790682,33519,monthly_arrears,Ramp depends on capacity allocation.


In [71]:
con.sql(
    """
    with cte_usage_first_info as
        (select customer_id,
                min(month) as first_month,
                max(month) as last_month
         from monthly_usage
         group by 1)

        , cte_billing_customer as
        (select distinct
                customer_id,
                segment,
                customer_status
         from billing_customers)

    select op.opportunity_id,
           op.crm_account_id,
           ac.matched_customer_id,
           op.probability,
           ac.matched_customer_id,
           date(date_trunc('month', op.expected_close_date)) as expected_close_date,
           date(date_trunc('month', op.expected_start_month)) as expected_start_month,
           frst.first_month,
           frst.last_month,
           op.stage,
           op.forecast_category,
           op.owner_forecast_notes,
           opportunity_type,
           stage,
           customer_status
    from crm_opportunities op
    left join crm_accounts ac
           on op.crm_account_id = ac.crm_account_id
    left join cte_usage_first_info frst
           on ac.matched_customer_id = frst.customer_id
    left join cte_billing_customer bc
           on frst.customer_id = bc.customer_id
    """).df()

,opportunity_id,crm_account_id,matched_customer_id,probability,matched_customer_id_1,expected_close_date,expected_start_month,first_month,last_month,stage,forecast_category,owner_forecast_notes,opportunity_type,stage_1,customer_status
0,OPP-1001,ACC-0002,CUST-0002,0.55,CUST-0002,2026-08-01,2026-09-01,2025-04-01,2026-05-01,Solution Fit,Upside,Legal review in progress.,New Logo,Solution Fit,active
1,OPP-1002,ACC-0008,CUST-0008,0.40,CUST-0008,2026-08-01,2026-09-01,2024-01-01,2026-05-01,Solution Fit,Upside,Legal review in progress.,Renewal,Solution Fit,active
2,OPP-1003,ACC-0009,CUST-0009,0.25,CUST-0009,2026-08-01,2026-10-01,2024-01-01,2026-05-01,Proposal,Best Case,Sales expects fast ramp.,New Logo,Proposal,active
3,OPP-1004,ACC-0012,CUST-0012,0.70,CUST-0012,2026-08-01,2026-09-01,2025-06-01,2026-05-01,Negotiation,Commit,Sales expects fast ramp.,Expansion,Negotiation,churned
4,OPP-1005,ACC-0014,CUST-0014,0.25,CUST-0014,2026-07-01,2026-07-01,2024-01-01,2025-10-01,Solution Fit,Pipeline,Sales expects fast ramp.,Expansion,Solution Fit,churned
5,OPP-1006,ACC-0015,CUST-0015,0.25,CUST-0015,2026-10-01,2026-11-01,2024-09-01,2026-05-01,Discovery,Pipeline,Legal review in progress.,Expansion,Discovery,active
6,OPP-1007,ACC-0016,CUST-0016,0.55,CUST-0016,2026-07-01,2026-09-01,2024-04-01,2026-05-01,Proposal,Commit,Procurement timing uncertain.,New Logo,Proposal,churned
7,OPP-1008,ACC-0019,CUST-0019,0.70,CUST-0019,2026-09-01,2026-10-01,2024-01-01,2026-05-01,Proposal,Best Case,Ramp depends on capacity allocation.,New Logo,Proposal,active
8,OPP-1009,ACC-0022,CUST-0022,0.40,CUST-0022,2026-08-01,2026-08-01,2024-01-01,2026-05-01,Commit,Best Case,Sales expects fast ramp.,Expansion,Commit,active
9,OPP-1010,ACC-0024,CUST-0024,0.25,CUST-0024,2026-09-01,2026-09-01,2024-01-01,2026-05-01,Proposal,Pipeline,Ramp depends on capacity allocation.,New Logo,Proposal,active


Interesting tables.
Let's take the example of ACC-0018 = Vertex Labs (from the crm_opportunities table).

1. In the crm_opportunities table they appear under crm_account_id = ACC-0018.
2. In the crm_accounts table, completely different records appear under the same crm_account_id. I have no choice but to use these tables, but in real life I would clarify this point.
3. Moreover, Vertex Labs appears in several rows in crm_accounts. How these tables map to each other is unclear.

Let's look at another peculiarity.
For our forecast period there are 3 opportunity_id values. There is a comment: Umbrella saving plan; avoid double counting child opps.

At the same time, the sum of the two opportunity_id values does not give the same result as the Umbrella saving plan.


In an ideal situation, I would clarify this with the team. But right now I need to make an assumption, so I will count only 1 row: OPP-1037. And I will remove the rows OPP-1035 and OPP-1036.

## Part 5: Invoices data

In [58]:
con.sql(
    """
    select *
    from invoices
    """).df()

,invoice_id,customer_id,invoice_date,service_month,contract_id,invoice_type,amount_usd,due_date,paid_date,payment_status
0,INV-70000,CUST-0001,2024-02-04,2024-01-01,NaN,usage,17405.99,2024-03-20,2024-03-20,paid
1,INV-70001,CUST-0001,2024-03-04,2024-02-01,NaN,usage,18012.21,2024-04-18,2024-04-18,paid
2,INV-70002,CUST-0001,2024-04-02,2024-03-01,NaN,usage,19299.41,2024-05-17,2024-06-04,paid
3,INV-70003,CUST-0001,2024-05-04,2024-04-01,NaN,usage,17041.90,2024-06-18,2024-07-06,paid
4,INV-70004,CUST-0001,2024-06-05,2024-05-01,NaN,usage,18738.35,2024-07-20,2024-08-07,paid
...,...,...,...,...,...,...,...,...,...,...
2157,INV-72155,CUST-0090,2026-06-05,2026-05-01,NaN,usage,19099.20,2026-08-04,2026-08-11,paid
2158,INV-72156,CUST-0018,2026-04-05,2026-03-01,NaN,service_credit,-215000.00,2026-04-05,2026-04-05,paid
2159,INV-72157,CUST-0033,2026-05-06,2026-04-01,NaN,billing_correction,-84000.00,2026-05-06,2026-05-06,paid
2160,INV-72158,CUST-0007,2026-01-05,2025-12-01,NaN,sla_credit,-36000.00,2026-01-05,2026-01-05,paid


## Key takeaways

What we know at the moment
1. monthly_usage looks like the table with the cleanest data. We will take it as the basis
2. There is a contracts table. Far from all customers from usage have contracts, and for some customers the contracts are finished. This gives information about: which discount applies and what the payment terms are.
3. There is also information about billing_customers. There is a churned status there, which looks similar to the real status.
4. There is also information about already-issued invoices. We will need this information to forecast cash receipts.
5. There is also a price-list table, with several price changes. One change is object_storage and it applies to everyone. The second change is the one that applies to new contracts and to renewing contracts.